# Setup & Imports

In [34]:
import pandas as pd
import numpy as np
import xgboost as xgb
import pickle
import os
from datetime import datetime, timedelta
from typing import List, Dict, Any
import warnings

warnings.filterwarnings('ignore') # Menghilangkan warning pandas yang mengganggu

# Konfigurasi Path File
FILE_RAW_EXCEL = "C:\\Kuliah\\TA\\models\\data.xlsx"
FILE_PROCESSED_CSV = "data_processed_clean.csv"
FILE_TRAINING_CSV = "training_data.csv"
MODEL_PATH = "xgboost_ranker_v6.pkl"

print("✅ Setup & Imports selesai.")

✅ Setup & Imports selesai.


# Preprocessing Data

In [35]:
print("Memulai preprocessing data...")

# 1. Load data
df = pd.read_excel(FILE_RAW_EXCEL)

# 2. Basic Cleaning
df['tambahan'] = df['tambahan'].fillna('').astype(str)
JENIS_COL = 'jenis_pakaian' 

# 3. Handle "Rok" dari kolom tambahan
rok_mask = df['tambahan'].str.contains('rok', case=False, regex=True)
rok_rows = df[rok_mask].copy()
rok_rows[JENIS_COL] = 'Rok'
df_processed = pd.concat([df, rok_rows], ignore_index=True)

# 4. Standarisasi jenis pakaian
def clean_jenis_pakaian(text):
    if pd.isna(text) or str(text).strip() == "":
        return "Kemeja" 
    
    t = str(text).lower().strip()
    
    if 'dinas' in t: return 'Dinas'
    if 'rok' in t: return 'Rok'
    if 'gamis' in t or 'kurung' in t: return 'Gamis'
    if 'kemeja' in t: return 'Kemeja'
    if 'gaun' in t: return 'Gaun'
    if 'basiba' in t: return 'Basiba'
    if 'kebaya' in t: return 'Kebaya'
    if 'blouse' in t: return 'Blouse'
    if 'blazer' in t or 'jas' in t: return 'Blazer'
    if 'rompi' in t: return 'Rompi'
    
    return t.title()

df_processed[JENIS_COL] = df_processed[JENIS_COL].apply(clean_jenis_pakaian)

# 5. One-Hot Encoding untuk "Tambahan"
def parse_tambahan(text):
    if not text or text.lower() == 'nan':
        return []
    return [p.strip().title() for p in text.split(',') if p.strip()]

all_tags = set()
df_processed['tambahan_list'] = df_processed['tambahan'].apply(parse_tambahan)
for tags in df_processed['tambahan_list']:
    all_tags.update(tags)

for tag in sorted(list(all_tags)):
    df_processed[tag] = df_processed['tambahan_list'].apply(lambda x: 1 if tag in x else 0)

# 6. Final Cleanup
cols_to_drop = [c for c in df_processed.columns if 'Lingkar' in c or 'Panjang' in c]
df_processed.drop(columns=cols_to_drop + ['tambahan', 'tambahan_list'], inplace=True)

# Simpan hasil
df_processed.to_csv(FILE_PROCESSED_CSV, index=False)
print(f"✅ Preprocessing selesai. Disimpan sebagai: {FILE_PROCESSED_CSV}")
print(f"Total baris: {len(df_processed)}, Total kolom: {len(df_processed.columns)}")

Memulai preprocessing data...
✅ Preprocessing selesai. Disimpan sebagai: data_processed_clean.csv
Total baris: 1218, Total kolom: 30


# Cek Fitur & EDA Sederhana

In [36]:
df_cek = pd.read_csv(FILE_PROCESSED_CSV)

tabel_data = df_cek['jenis_pakaian'].value_counts().reset_index()
tabel_data.columns = ['Jenis Pakaian', 'Jumlah (pcs)']

print("="*40)
print("      LAPORAN JUMLAH JENIS PAKAIAN")
print("="*40)
print(tabel_data.to_string(index=False)) # Print rapi tanpa index
print("="*40)
print(f" TOTAL PESANAN: {tabel_data['Jumlah (pcs)'].sum()} pcs\n")

      LAPORAN JUMLAH JENIS PAKAIAN
Jenis Pakaian  Jumlah (pcs)
          Rok           337
        Dinas           273
       Basiba           220
       Blouse           129
        Gamis           125
       Kebaya            63
       Blazer            43
       Kemeja            18
         Gaun             8
        Rompi             2
 TOTAL PESANAN: 1218 pcs



# Ranking Logic & Urgency Config

In [37]:
ALPHA = 0.1  # Urgency decay factor

# Label encoding untuk jenis pakaian (agar jadi fitur numerik untuk XGBoost)
JENIS_MAPPING = {
    'Dinas': 0, 'Rok': 1, 'Gamis': 2, 'Basiba': 3, 'Blouse': 4,
    'Kebaya': 5, 'Blazer': 6, 'Kemeja': 7, 'Gaun': 8, 'Rompi': 9
}

class Order:
    def __init__(self, order_id: int, nama_pelanggan: str, jenis_pakaian: str, deadline: datetime, features: Dict[str, int] = None):
        self.id = order_id
        self.nama_pelanggan = nama_pelanggan
        self.jenis_pakaian = jenis_pakaian
        self.deadline = deadline
        self.features = features if features is not None else {}
        
    @property
    def complexity_score(self) -> float:
        return sum(self.features.values())

def calculate_urgency(deadline: datetime, current_date: datetime = None) -> float:
    """Hitung skor urgensi berdasarkan jarak ke deadline (Exponential Decay)."""
    if current_date is None:
        current_date = datetime.now()
    
    days_remaining = (deadline - current_date).days
    
    # Jika sudah lewat deadline (minus), urgensi maksimal
    if days_remaining < 0:
        return 1.0 
        
    return float(np.exp(-ALPHA * days_remaining))

print("✅ Ranking logic (Order Class & Urgency) dimuat.")

✅ Ranking logic (Order Class & Urgency) dimuat.


# Data Generator

In [38]:
def generate_training_data(input_csv, output_csv):
    print(f"Loading data from {input_csv}...")
    df = pd.read_csv(input_csv)
    
    # Parsing Tanggal
    df['tanggal_masuk'] = pd.to_datetime(df['tanggal_masuk'], errors='coerce')
    
    df['deadline'] = pd.to_datetime(df['deadline'], errors='coerce')
    df = df.dropna(subset=['tanggal_masuk', 'deadline'])
    
    # Buat "Group/Batch" (Pesanan yang masuk di minggu yang sama akan saling diadu)
    df['batch_group'] = df['tanggal_masuk'].dt.strftime('%Y-W%U') # Format: Tahun-MingguKe
    
    # Hitung Durasi Pengerjaan Nyata (Lead Time)
    df['lead_time_days'] = (df['deadline'] - df['tanggal_masuk']).dt.days
    df['lead_time_days'] = df['lead_time_days'].clip(lower=1) # Hindari pembagian dengan 0
    
    # Hitung Kerumitan
    main_columns = ['tanggal_masuk', 'deadline', 'nama_pelanggan', 'jenis_pakaian', 'batch_group', 'lead_time_days']
    feature_cols = [col for col in df.columns if col not in main_columns]
    df['complexity_score'] = df[feature_cols].sum(axis=1)
    
    # Label encode jenis_pakaian sebagai fitur numerik untuk XGBoost
    df['jenis_encoded'] = df['jenis_pakaian'].map(JENIS_MAPPING)
    
    # Logika: Jika baju rumit diselesaikan dengan cepat -> Prioritas historisnya tinggi
    df['historical_priority_score'] = df['complexity_score'] / df['lead_time_days']
    
    # XGBRanker diurutkan berdasarkan Group
    df = df.sort_values('batch_group').reset_index(drop=True)
    
    df.to_csv(output_csv, index=False)
    print(f"Training data generated: {output_csv} ({len(df)} records)")
    return df

# Jalankan generator
df_train = generate_training_data(FILE_PROCESSED_CSV, FILE_TRAINING_CSV)

Loading data from data_processed_clean.csv...
Training data generated: training_data.csv (1205 records)


# Train XGBoost Ranker

In [ ]:
# Train XGBoost Ranker v6 (Optimized & Time-Aware)
def prepare_features_and_labels_v6(df):
    df = df.sort_values('batch_group').reset_index(drop=True)
    
    # Pastikan format tanggal benar
    df['tanggal_masuk'] = pd.to_datetime(df['tanggal_masuk'], errors='coerce')
    df['deadline'] = pd.to_datetime(df['deadline'], errors='coerce')
    
    # Hitung jarak hari (Bisa bernilai negatif jika telat)
    df['days_to_deadline'] = (df['deadline'] - df['tanggal_masuk']).dt.days
    
    # Jika telat (negatif), berikan denda tinggi secara linear agar Rank naik drastis
    penalty_telat = np.where(df['days_to_deadline'] < 0, abs(df['days_to_deadline']) * 2.0, 0)
    
    # Bobot 2.5 agar tidak menenggelamkan skor kerumitan
    urgency_multiplier = np.exp(-0.1 * df['days_to_deadline'].clip(lower=0)) * 2.5
    
    # Gabungkan: Sejarah + Urgensi + Denda Telat
    df['target_kombinasi'] = df['historical_priority_score'] + urgency_multiplier + penalty_telat
    
    ignore_cols = [
        'tanggal_masuk', 'deadline', 'nama_pelanggan', 
        'batch_group', 'lead_time_days', 'historical_priority_score', 'target_kombinasi', 'order_id'
    ]
    
    feature_cols = [col for col in df.columns if col not in ignore_cols and df[col].dtype in ['int64', 'float64', 'int32']]
    
    X = df[feature_cols]
    y = df['target_kombinasi'].values 
    groups = df.groupby('batch_group').size().values 
    
    return X, y, groups, feature_cols

print(f"Loading training data dari {FILE_TRAINING_CSV}...")
df_train = pd.read_csv(FILE_TRAINING_CSV)

X, y, groups, feature_cols = prepare_features_and_labels_v6(df_train)

dtrain = xgb.DMatrix(X, label=y)
dtrain.set_group(groups)

# Gunakan parameter terbaik dari Grid Search
params = {
    'objective': 'rank:pairwise',
    'eval_metric': 'ndcg',
    'seed': 42,
    'colsample_bytree': 0.8,
    'eta': 0.1,
    'max_depth': 6,
    'subsample': 0.8
}

print("Training XGBoost Ranker v6 (Time-Aware)...")
model_v6 = xgb.train(params, dtrain, num_boost_round=150, verbose_eval=False)
model_v6.feature_names_in_ = feature_cols

with open(MODEL_PATH, 'wb') as f:
    pickle.dump(model_v6, f)
    
print(f"Model v6     berhasil dilatih dan disimpan ke: {MODEL_PATH}")

Loading training data dari training_data.csv...
Training XGBoost Ranker v6 (Time-Aware)...
Model v6     berhasil dilatih dan disimpan ke: xgboost_ranker_v6.pkl


# Testing & Inference

In [40]:
def get_color_label(urgency_score: float) -> str:
    if urgency_score > 0.8: return "🔴 Red"
    elif urgency_score > 0.5: return "🟡 Yellow"
    else: return "🟢 Green"
    
def prepare_inference_data_v6(orders: List[Order], current_date: datetime) -> pd.DataFrame:
    data = []
    for order in orders:
        row = {
            'order_id': order.id,
            'jenis_encoded': JENIS_MAPPING.get(order.jenis_pakaian, -1),
            'complexity_score': order.complexity_score,
            'urgency_score': calculate_urgency(order.deadline, current_date),
            'days_to_deadline': (order.deadline - current_date).days, 
        }
        row.update(order.features) 
        data.append(row)
    return pd.DataFrame(data)

def rank_orders_v6(orders: List[Order], current_date: datetime) -> List[Dict[str, Any]]:
    if not orders: return []
    df = prepare_inference_data_v6(orders, current_date)
    
    try:
        with open(MODEL_PATH, 'rb') as f:
            loaded_model = pickle.load(f)
        
        expected_features = getattr(loaded_model, 'feature_names_in_', [])
        for col in expected_features:
            if col not in df.columns: df[col] = 0
                
        X = df[expected_features]
        df['model_score'] = loaded_model.predict(xgb.DMatrix(X))
        df = df.sort_values('model_score', ascending=False)
        
    except Exception as e:
        print(f"⚠️ Error model: {e}. Fallback ke manual urgency.")
        df = df.sort_values('urgency_score', ascending=False)
        df['model_score'] = 0.0
    
    results = []
    for idx, (_, row) in enumerate(df.iterrows(), start=1):
        order = next(o for o in orders if o.id == row['order_id'])
        results.append({
            'Rank': idx,
            'ID': order.id,
            'Pelanggan': order.nama_pelanggan,
            'Pakaian': order.jenis_pakaian,
            'Status': get_color_label(row['urgency_score']), # Asumsi fungsi get_color_label sudah ada dari V3
            'Sisa Hari': row['days_to_deadline'],
            'Skor AI': round(row['model_score'], 4) 
        })
    return results

# --- SIMULASI ---
print("\n⚙️ Mensimulasikan Antrean Pesanan Masuk (Model v6 - Time-Aware)...")
sim_now = datetime.now()

dummy_orders_v6 = [
    Order(101, "Ahmad", "Kemeja", sim_now + timedelta(days=2), {'kancing': 1}), 
    Order(102, "Budi", "Jas", sim_now + timedelta(days=14), {'furing': 1, 'bordir': 1}), 
    Order(103, "Citra", "Gamis", sim_now - timedelta(days=1), {'payet': 1, 'resleting': 1}), 
]

hasil_ranking_v6 = rank_orders_v6(dummy_orders_v6, sim_now)
display(pd.DataFrame(hasil_ranking_v6   ))


⚙️ Mensimulasikan Antrean Pesanan Masuk (Model v6 - Time-Aware)...


,Rank,ID,Pelanggan,Pakaian,Status,Sisa Hari,Skor AI
0,1,103,Citra,Gamis,🔴 Red,-1.0,4.4800
1,2,101,Ahmad,Kemeja,🔴 Red,2.0,4.2496
2,3,102,Budi,Jas,🟢 Green,14.0,-0.9983


# Evaluasi

In [41]:
# Evaluasi Akhir v6
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

print("Memulai Evaluasi Model Versi 4 (Time-Aware)...")

TARGET_COL = 'target_kombinasi'
GROUP_COL = 'batch_group'

df_eval = df_train.copy()

# Bentuk ulang target karena ini DataFrame duplikat untuk testing
df_eval['tanggal_masuk'] = pd.to_datetime(df_eval['tanggal_masuk'], errors='coerce')
df_eval['deadline'] = pd.to_datetime(df_eval['deadline'], errors='coerce')
df_eval['days_to_deadline'] = (df_eval['deadline'] - df_eval['tanggal_masuk']).dt.days
urgency_multiplier = np.exp(-0.1 * df_eval['days_to_deadline'].clip(lower=0))
df_eval['target_kombinasi'] = df_eval['historical_priority_score'] + (urgency_multiplier * 5)

ignore_cols = ['tanggal_masuk', 'deadline', 'nama_pelanggan', 'jenis_pakaian', 
               'batch_group', 'lead_time_days', 'historical_priority_score', 'target_kombinasi', 'order_id']
feature_cols = [col for col in df_eval.columns if col not in ignore_cols and df_eval[col].dtype in ['int64', 'float64', 'int32']]

# Split Data Test vs Train
gss = GroupShuffleSplit(test_size=0.20, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(df_eval[feature_cols], df_eval[TARGET_COL], groups=df_eval[GROUP_COL]))

train_df = df_eval.iloc[train_idx].sort_values(GROUP_COL).copy()
test_df = df_eval.iloc[test_idx].sort_values(GROUP_COL).copy()

# Scale Target
scaler = MinMaxScaler(feature_range=(0, 10))
y_train_int = np.clip(np.round(scaler.fit_transform(train_df[[TARGET_COL]])), 0, 10).astype(int).flatten()
y_test_int = np.clip(np.round(scaler.transform(test_df[[TARGET_COL]])), 0, 10).astype(int).flatten()

dtrain = xgb.DMatrix(train_df[feature_cols], label=y_train_int)
dtrain.set_group(train_df.groupby(GROUP_COL, sort=False).size().values)
dtest = xgb.DMatrix(test_df[feature_cols], label=y_test_int)
dtest.set_group(test_df.groupby(GROUP_COL, sort=False).size().values)

# Gunakan best parameter dari V3
params = {'objective': 'rank:pairwise', 'eval_metric': 'ndcg', 'seed': 42, 
          'colsample_bytree': 0.8, 'eta': 0.1, 'max_depth': 6, 'subsample': 0.8}

evals_result = {}
model_eval = xgb.train(params, dtrain, num_boost_round=150, evals=[(dtest, 'test')], evals_result=evals_result, verbose_eval=False)

ndcg_score = evals_result['test']['ndcg'][-1]

print("\n" + "="*40)
print("     LAPORAN EVALUASI MODEL VERSI 4")
print("="*40)
print(f"NDCG Score        : {ndcg_score:.4f} ")
print(f"Total Fitur       : {len(feature_cols)} (Termasuk days_to_deadline)")
print("="*40)

Memulai Evaluasi Model Versi 4 (Time-Aware)...

     LAPORAN EVALUASI MODEL VERSI 4
NDCG Score        : 0.9829 
Total Fitur       : 29 (Termasuk days_to_deadline)


In [42]:
# =============================================
# FIXED: MAP & Pairwise Accuracy
# =============================================
from sklearn.metrics import average_precision_score

def compute_map(y_true_groups, y_pred_groups):
    """
    MAP yang benar:
    - Urutkan berdasarkan y_pred (descending)
    - Hitung AP per group dari urutan tersebut
    - Relevan = label >= median group
    """
    ap_scores = []
    for y_true, y_pred in zip(y_true_groups, y_pred_groups):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        threshold = np.median(y_true)
        y_binary = (y_true >= threshold).astype(int)

        # Skip jika semua sama (tidak ada variasi relevansi)
        if y_binary.sum() == 0 or y_binary.sum() == len(y_binary):
            continue

        # FIX: AP dihitung dari urutan prediksi, bukan linspace
        # average_precision_score(y_true, y_score) → y_score menentukan urutan
        ap = average_precision_score(y_binary, y_pred)
        ap_scores.append(ap)

    return np.mean(ap_scores) if ap_scores else 0.0


def compute_pairwise_accuracy(y_true_groups, y_pred_groups):
    correct = 0
    total = 0
    for y_true, y_pred in zip(y_true_groups, y_pred_groups):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        n = len(y_true)
        for i in range(n):
            for j in range(i + 1, n):
                if y_true[i] != y_true[j]:
                    total += 1
                    if (y_true[i] > y_true[j]) == (y_pred[i] > y_pred[j]):
                        correct += 1
    return correct / total if total > 0 else 0.0


# --- Ambil prediksi dari model yang sudah ada ---
# PENTING: gunakan dtest yang sudah ada (test split), BUKAN dtrain
pred_scores = model_eval.predict(dtest)

# --- Kelompokkan per group dari test_df ---
y_true_groups = []
y_pred_groups = []

start = 0
group_sizes = test_df.groupby(GROUP_COL, sort=False).size().values
for size in group_sizes:
    end = start + size
    y_true_groups.append(y_test_int[start:end].tolist())
    y_pred_groups.append(pred_scores[start:end].tolist())
    start = end

# --- Hitung ---
map_score = compute_map(y_true_groups, y_pred_groups)
pairwise_acc = compute_pairwise_accuracy(y_true_groups, y_pred_groups)

print("\n" + "="*40)
print("   LAPORAN EVALUASI LENGKAP MODEL V6")
print("="*40)
print(f"NDCG Score        : {ndcg_score:.4f}")
print(f"MAP Score         : {map_score:.4f}")
print(f"Pairwise Accuracy : {pairwise_acc:.4f}  ({pairwise_acc*100:.2f}%)")
print("="*40)


   LAPORAN EVALUASI LENGKAP MODEL V6
NDCG Score        : 0.9829
MAP Score         : 1.0000
Pairwise Accuracy : 0.9899  (98.99%)


In [43]:
# === DIAGNOSTIK MAP ===
import matplotlib.pyplot as plt

print("=== CEK DISTRIBUSI y_test_int ===")
unique, counts = np.unique(y_test_int, return_counts=True)
print(dict(zip(unique, counts)))

print(f"\nTotal test samples : {len(y_test_int)}")
print(f"Total groups       : {len(group_sizes)}")
print(f"Group sizes        : min={group_sizes.min()}, max={group_sizes.max()}, mean={group_sizes.mean():.1f}")

print("\n=== CEK 5 GROUP PERTAMA ===")
start = 0
for i, size in enumerate(group_sizes[:5]):
    end = start + size
    yt = y_test_int[start:end]
    yp = pred_scores[start:end]
    threshold = np.median(yt)
    y_bin = (yt >= threshold).astype(int)
    print(f"Group {i}: size={size} | y_true={yt} | y_binary={y_bin} | y_pred={np.round(yp,2)}")
    start = end

print("\n=== CEK BERAPA GROUP YANG DI-SKIP ===")
skipped = 0
included = 0
start = 0
for size in group_sizes:
    end = start + size
    yt = np.array(y_test_int[start:end])
    threshold = np.median(yt)
    y_bin = (yt >= threshold).astype(int)
    if y_bin.sum() == 0 or y_bin.sum() == len(y_bin):
        skipped += 1
    else:
        included += 1
    start = end

print(f"Group di-skip  : {skipped}")
print(f"Group dihitung : {included}")

=== CEK DISTRIBUSI y_test_int ===
{np.int64(0): np.int64(5), np.int64(1): np.int64(48), np.int64(2): np.int64(72), np.int64(3): np.int64(43), np.int64(4): np.int64(34), np.int64(5): np.int64(11), np.int64(6): np.int64(1), np.int64(7): np.int64(2), np.int64(8): np.int64(1)}

Total test samples : 217
Total groups       : 5
Group sizes        : min=23, max=72, mean=43.4

=== CEK 5 GROUP PERTAMA ===
Group 0: size=38 | y_true=[3 4 4 4 2 3 3 3 2 4 2 4 2 2 4 4 2 3 3 4 4 3 4 4 4 2 2 4 4 2 2 2 2 2 2 4 4
 4] | y_binary=[1 1 1 1 0 1 1 1 0 1 0 1 0 0 1 1 0 1 1 1 1 1 1 1 1 0 0 1 1 0 0 0 0 0 0 1 1
 1] | y_pred=[ 0.84  2.98  2.56  2.19 -1.56  0.35  0.81  0.79 -1.62  1.32 -1.91  2.72
 -1.62 -3.15  2.03  2.03 -1.62  0.87  0.65  2.05  2.76  0.8   3.02  2.88
  2.03 -2.11 -2.    2.05  2.03 -3.15 -2.06 -2.06 -2.06 -2.06 -2.04  2.17
  3.11  1.95]
Group 1: size=50 | y_true=[3 3 3 0 3 3 3 0 8 3 3 3 3 1 1 4 3 1 3 3 3 3 5 4 5 3 5 1 5 4 0 5 1 1 7 6 7
 1 5 4 3 4 4 1 0 5 5 5 1 0] | y_binary=[1 1 1 0 1 1 1 0 1 1 1 1

In [44]:
# === FIX: MAP dengan Top-K Relevance, bukan median ===

def compute_map_topk(y_true_groups, y_pred_groups, k_ratio=0.25):
    """
    Relevan = top k% label tertinggi per group.
    Lebih diskriminatif dari median.
    """
    ap_scores = []
    for y_true, y_pred in zip(y_true_groups, y_pred_groups):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        
        # Top 25% tertinggi = relevan
        k = max(1, int(len(y_true) * k_ratio))
        threshold = np.sort(y_true)[::-1][k - 1]
        y_binary = (y_true >= threshold).astype(int)
        
        # Hindari semua-1 atau semua-0
        if y_binary.sum() == 0 or y_binary.sum() == len(y_binary):
            continue
        
        ap = average_precision_score(y_binary, y_pred)
        ap_scores.append(ap)
    
    return np.mean(ap_scores) if ap_scores else 0.0


def compute_map_per_query(y_true_groups, y_pred_groups):
    """
    MAP versi strict: hanya item dengan label = MAX per group yang relevan.
    Paling ketat, paling informatif untuk ranking task.
    """
    ap_scores = []
    for y_true, y_pred in zip(y_true_groups, y_pred_groups):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        
        # Hanya label maksimum = relevan
        max_label = y_true.max()
        y_binary = (y_true == max_label).astype(int)
        
        if y_binary.sum() == 0 or y_binary.sum() == len(y_binary):
            continue
        
        ap = average_precision_score(y_binary, y_pred)
        ap_scores.append(ap)
    
    return np.mean(ap_scores) if ap_scores else 0.0


# --- Jalankan ketiganya ---
map_median   = compute_map(y_true_groups, y_pred_groups)       # lama
map_topk     = compute_map_topk(y_true_groups, y_pred_groups)  # top 25%
map_strict   = compute_map_per_query(y_true_groups, y_pred_groups)  # hanya max

print("\n" + "="*45)
print("   LAPORAN EVALUASI LENGKAP MODEL V6")
print("="*45)
print(f"NDCG Score         : {ndcg_score:.4f}")
print(f"MAP                : {map_strict:.4f}")
print(f"Pairwise Accuracy  : {pairwise_acc:.4f}  ({pairwise_acc*100:.2f}%)")
print("="*45)


   LAPORAN EVALUASI LENGKAP MODEL V6
NDCG Score         : 0.9829
MAP                : 0.9000
Pairwise Accuracy  : 0.9899  (98.99%)
